## [ 과제 2 ] 웹캠으로 가위바위보 인식

In [2]:
## -------------------------------------
## 이미지 데이터 로딩 & 기본 정보
## -------------------------------------
import cv2
import os
import matplotlib.pyplot as plt
import koreanize_matplotlib
import numpy as np
import matplotlib.cm as cm
import sys
sys.path.append(r"C:\KDT14\7_CV")
import math
import importlib
import cv_utils

importlib.reload(cv_utils)

<module 'cv_utils' from 'c:\\Users\\82105\\OneDrive\\바탕 화면\\14_ai\\work_0616_이한이\\cv_utils.py'>

In [16]:
cap = cv2.VideoCapture(0)
kernel = np.ones((5, 5), np.uint8)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 거울처럼 보이게 좌우 반전
    frame = cv2.flip(frame, 1)

    # 손 인식할 ROI 박스 설정
    x1, y1 = 100, 100
    x2, y2 = 450, 450

    roi = frame[y1:y2, x1:x2]
    roi_draw = roi.copy()

    # 그레이 변환 후 블러로 노이즈 줄이기
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (7, 7), 0)

    # 밝은 배경에서 손만 추출 (역이진화)
    _, th = cv2.threshold(blur, 180, 255, cv2.THRESH_BINARY_INV)

    # 작은 노이즈 제거하고 구멍 메우기
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel)

    # 외곽 컨투어만 찾기
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2:]

    gesture = "No Hand"
    defect_count = 0

    if len(contours) > 0:
        # 가장 큰 덩어리 = 손
        cntr = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cntr)

        if area > 5000:  # 너무 작은 건 무시
            hull = cv2.convexHull(cntr)
            cv2.drawContours(roi_draw, [cntr], -1, (0, 255, 0), 2)   # 손 윤곽 초록
            cv2.drawContours(roi_draw, [hull], -1, (255, 0, 0), 2)   # 볼록 껍질 파랑

            # defect 계산용 hull (인덱스 기반)
            hull2 = cv2.convexHull(cntr, returnPoints=False)
            if hull2 is not None and len(hull2) > 3:
                defects = cv2.convexityDefects(cntr, hull2)
                if defects is not None:
                    for i in range(defects.shape[0]):
                        startP, endP, farthestP, distance = defects[i, 0]
                        start = tuple(cntr[startP][0])
                        end = tuple(cntr[endP][0])
                        farthest = tuple(cntr[farthestP][0])

                        # 골짜기가 충분히 깊고, 손목 아래쪽 노이즈는 제외
                        if distance > 8000 and farthest[1] < roi.shape[0] * 0.85:
                            defect_count += 1
                            cv2.circle(roi_draw, farthest, 6, (0, 0, 255), -1)  # 골짜기 빨간 점
                            cv2.line(roi_draw, start, end, (0, 255, 255), 2)    # 손가락 끝 연결선

            # 골짜기 수로 제스처 판별 (손가락 수 = defect + 1)
            if defect_count >= 4:
                gesture = "paper"
            elif defect_count == 1:
                gesture = "Scissor"
            elif defect_count == 0:
                gesture = "Rock"
            else:
                gesture = "Unknown"

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(frame, f"Gesture: {gesture}",      (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    cv2.putText(frame, f"Defects: {defect_count}", (30, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    cv2.imshow("Frame", frame)
    cv2.imshow("ROI", roi_draw)
    cv2.imshow("Binary", th)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()